In [1]:
import torch
import tensorviz as tv
from demo import attention

In [ ]:
x = torch.randn(2, 5, 32, device="meta")  # meta: shapes only, no compute
W = [torch.randn(32, 32, device="meta") for _ in range(3)]

with tv.Trace() as t:
    out = attention(x, *W, H=4)
t

In [ ]:
# code defined in the cell itself must trace too
with tv.Trace() as t2:
    a = torch.randn(3, 4, device="meta")
    b = torch.randn(4, 5, device="meta")
    c = a @ b
t2

## ViT patchify: image (B,C,H,W) -> tokens (B,N+1,D)

In [ ]:
B, C, H, W, P, D = 2, 3, 224, 224, 16, 768   # batch, channels, height, width, patch size, embed dim
G = H // P                # patches per side
N = G * G                 # number of patches
K = P * P * C             # flattened patch length
img  = torch.randn(B, C, H, W, device="meta")
proj = torch.nn.Linear(K, D, device="meta")
cls  = torch.randn(1, 1, D, device="meta")
pos  = torch.randn(1, N + 1, D, device="meta")

with tv.Trace() as vit:   # UPPERCASE ints above label the dims; extra: tv.Trace(Z=123)
    grid    = img.reshape(B, C, G, P, G, P)          # cut H and W into (G, P) each
    perm    = grid.permute(0, 2, 4, 3, 5, 1)         # move the two patch-grid axes next to B
    patches = perm.reshape(B, N, K)                  # flatten each P x P x C patch -> N x K
    tokens  = proj(patches)                          # linear embed -> N x D
    seq     = torch.cat([cls.expand(B, -1, -1), tokens], dim=1)  # prepend [CLS] -> N+1
    z       = seq + pos                              # add position embedding
vit

## Shape bug: the failing op is highlighted
If an op fails inside the block, the trace is displayed right away (the cell's trailing `t` would never run), then the error still propagates.

In [ ]:
B, T, D = 2, 5, 32
x = torch.randn(B, T, D, device="meta")
W_bad = torch.randn(64, 16, device="meta")  # should start with D = 32

try:
    with tv.Trace():
        a = x + 1
        h = a @ W_bad  # 32 != 64
except RuntimeError as e:
    print("error still propagates:", str(e).splitlines()[0])